# Projekt · Woche 8 · Tools für euren Assistenten
**AI Engineering: von LLMs zu Agenten** · linkit e.V. am KIT

Euer Assistent bekommt diese Woche Werkzeuge. Für beide Tracks sind Beispiel-Tools fertig programmiert, darunter jeweils ein **schreibendes** Tool mit Bestätigung. Eure Aufgabe:

1. Die Beispiel-Tools verstehen und ausprobieren
2. **Mindestens ein Tool anpassen oder ein eigenes ergänzen**, passend zu euren User Stories
3. Eure Testfragen aus dem Steckbrief stellen und notieren, welche jetzt beantwortet werden
4. Im Team-Repo speichern

Das Notebook erkennt am Startpaket, welcher Track gemeint ist.

In [ ]:
%pip install -q sentence-transformers==6.1.0 openai==3.24.0

In [ ]:
import os, glob, zipfile, csv, json, re, datetime
import numpy as np

ORDNER = "daten"
if not os.path.isdir(ORDNER):
    from google.colab import files
    hochgeladen = files.upload()
    os.makedirs(ORDNER, exist_ok=True)
    for name in hochgeladen:
        if name.endswith(".zip"):
            zipfile.ZipFile(name).extractall(ORDNER)
        else:
            os.replace(name, os.path.join(ORDNER, name))

TRACK = "B" if os.path.exists(f"{ORDNER}/wartungshandbuch.md") else "A"
HEUTE = datetime.date.today()
print("Track", TRACK, "| heute:", HEUTE)

def lies_csv(name):
    with open(f"{ORDNER}/{name}", encoding="utf-8") as f:
        return list(csv.DictReader(f))

## 1 · Wissensbasis als Tool

Nur die Textdokumente kommen in die Wissensbasis. Die Tabellen erreicht der Assistent über eigene Tools, damit er exakte Werte bekommt und personenbezogene Daten nur über klar geregelte Wege.

In [ ]:
from sentence_transformers import SentenceTransformer

def chunk_struktur(markdown, dokument):
    chunks, pfad, zeilen, titel = [], {}, [], dokument
    def abschliessen():
        inhalt = "\n".join(zeilen).strip()
        if inhalt:
            chunks.append({"titel": titel, "text": inhalt})
    for zeile in markdown.splitlines():
        if zeile.startswith("#"):
            abschliessen()
            ebene = len(zeile) - len(zeile.lstrip("#"))
            pfad = {e: t for e, t in pfad.items() if e < ebene}
            pfad[ebene] = zeile.lstrip("#").strip()
            titel = dokument + " › " + " › ".join(pfad[e] for e in sorted(pfad))
            zeilen = []
        else:
            zeilen.append(zeile)
    abschliessen()
    return chunks

CHUNKS = []
for d in sorted(glob.glob(f"{ORDNER}/**/*.md", recursive=True)):
    if not os.path.basename(d).lower().startswith("readme"):
        CHUNKS += chunk_struktur(open(d, encoding="utf-8").read(), os.path.basename(d))
modell = SentenceTransformer("intfloat/multilingual-e5-small")
E = modell.encode(["passage: " + c["titel"] + "\n" + c["text"] for c in CHUNKS], normalize_embeddings=True)
print(len(CHUNKS), "Chunks in der Wissensbasis")

def search_knowledge_base(query, k=4):
    q = modell.encode(["query: " + query], normalize_embeddings=True)[0]
    return [{"quelle": CHUNKS[i]["titel"], "text": CHUNKS[i]["text"]} for i in np.argsort(E @ q)[::-1][:k]]

THEMEN = "Wartungshandbuch der Nordwerk GmbH (Intervalle, Sicherheit, Störungsbehebung, Betriebsstoffe, Ersatzteile, Fehlercodes)" if TRACK == "B" \
         else "Satzung und Sitzungsprotokolle der Datenfabrik Karlsruhe e.V. (Regeln, Beschlüsse, Budgets, Zuständigkeiten)"
TOOL_SUCHE = {
    "name": "search_knowledge_base",
    "description": f"Durchsucht {THEMEN} und liefert die passendsten Abschnitte mit Quelle. Nutze es für Fragen zu Regeln, Abläufen und dokumentiertem Wissen.",
    "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "Suchanfrage in natürlicher Sprache."}}, "required": ["query"]},
}

## 2 · Domänen-Tools

Die Zelle legt je nach Track drei Tools an. **Schreibende Tools** stehen in `SCHREIBENDE_TOOLS` und werden nur nach Bestätigung ausgeführt.

In [ ]:
FUNKTIONEN = {"search_knowledge_base": search_knowledge_base}
TOOLS = [TOOL_SUCHE]
SCHREIBENDE_TOOLS = set()

if TRACK == "B":
    MASCHINEN = {m["id"]: m for m in lies_csv("maschinen.csv")}
    TICKETS = lies_csv("stoerungen.csv")
    TICKET_LOG = []

    def _pruefe(machine_id):
        if machine_id not in MASCHINEN:
            raise ValueError(f"Unbekannte Maschinen-ID '{machine_id}'. Gültig: {', '.join(MASCHINEN)}.")

    def get_machine_status(machine_id):
        _pruefe(machine_id)
        m = MASCHINEN[machine_id]
        offen = sum(t["maschine"] == machine_id and t["status"] == "offen" for t in TICKETS)
        return {**m, "offene_tickets": offen}

    def list_tickets(machine_id=None, status="offen"):
        if machine_id: _pruefe(machine_id)
        return [t for t in TICKETS + TICKET_LOG if (machine_id is None or t["maschine"] == machine_id) and (status == "alle" or t["status"] == status)]

    def create_ticket(machine_id, description, priority):
        _pruefe(machine_id)
        ticket = {"ticket": f"T-{1001 + len(TICKET_LOG)}", "datum": HEUTE.isoformat(), "maschine": machine_id,
                  "prioritaet": priority, "status": "offen", "beschreibung": description}
        TICKET_LOG.append(ticket)
        return {"ticket": ticket["ticket"], "status": "angelegt"}

    ID_TEXT = "Maschinen-ID im Format M-01 bis M-08."
    FUNKTIONEN |= {"get_machine_status": get_machine_status, "list_tickets": list_tickets, "create_ticket": create_ticket}
    TOOLS += [
        {"name": "get_machine_status", "description": "Aktueller Zustand einer Maschine: Status, Betriebsstunden, letzte Wartung, Standort, offene Tickets.",
         "parameters": {"type": "object", "properties": {"machine_id": {"type": "string", "description": ID_TEXT}}, "required": ["machine_id"], "additionalProperties": False}},
        {"name": "list_tickets", "description": "Listet Störungstickets, standardmäßig nur offene. Optional für eine Maschine; status 'alle' zeigt auch geschlossene.",
         "parameters": {"type": "object", "properties": {"machine_id": {"type": "string", "description": ID_TEXT}, "status": {"type": "string", "enum": ["offen", "geschlossen", "alle"]}}}},
        {"name": "create_ticket", "description": "Legt ein neues Störungsticket an. Nur verwenden, wenn der Nutzer ausdrücklich ein Ticket möchte. Prioritäten: hoch = Maschine steht oder Sicherheitsrisiko, mittel = eingeschränkt, niedrig = kein Einfluss.",
         "parameters": {"type": "object", "properties": {"machine_id": {"type": "string", "description": ID_TEXT}, "description": {"type": "string"}, "priority": {"type": "string", "enum": ["hoch", "mittel", "niedrig"]}},
                          "required": ["machine_id", "description", "priority"], "additionalProperties": False}},
    ]
    SCHREIBENDE_TOOLS = {"create_ticket"}

else:
    TERMINE = lies_csv("termine.csv")
    MITGLIEDER = {m["id"]: m for m in lies_csv("mitglieder.csv")}

    def _semester_index(text):
        jahr = int(re.search(r"\d{4}", text).group())
        return jahr * 2 + (1 if text.startswith("WS") else 0)

    def list_events(ab_datum=None, art=None):
        ab = ab_datum or HEUTE.isoformat()
        return [t for t in TERMINE if t["datum"] >= ab and (art is None or t["art"].lower() == art.lower())][:8]

    def get_member_engagement(member_id):
        if member_id not in MITGLIEDER:
            raise ValueError(f"Unbekannte Mitglieds-ID '{member_id}'. Format: D-001.")
        m = MITGLIEDER[member_id]
        aktuell = _semester_index(f"WS {HEUTE.year}" if HEUTE.month >= 10 else f"SoSe {HEUTE.year}")
        # Bewusst nur die Daten, die für eine Bescheinigung nötig sind: keine E-Mail, kein Studiengang
        return {"id": m["id"], "name": f"{m['vorname']} {m['nachname']}".strip(), "rolle": m["rolle"], "ressort": m["ressort"],
                "eintritt": m["eintritt"], "semester_seit_eintritt": aktuell - _semester_index(m["eintritt"]) + 1,
                "engagement_stunden": int(m["engagement_stunden"]), "status": m["status"]}

    def draft_certificate(member_id, taetigkeiten):
        d = get_member_engagement(member_id)
        os.makedirs("bescheinigungen", exist_ok=True)
        pfad = f"bescheinigungen/{member_id}_entwurf.md"
        with open(pfad, "w", encoding="utf-8") as f:
            f.write(f"# Engagementbescheinigung (Entwurf)\n\n{d['name']} ist seit dem {d['eintritt']} Mitglied der Datenfabrik Karlsruhe e.V. "
                    f"und hat sich in dieser Zeit mit rund {d['engagement_stunden']} Stunden engagiert, zuletzt als {d['rolle']}.\n\n"
                    f"Wesentliche Tätigkeiten: {taetigkeiten}\n\nKarlsruhe, {HEUTE:%d.%m.%Y}\n\n(Unterschrift zweier Vorstandsmitglieder)\n")
        return {"entwurf": pfad, "status": "angelegt"}

    FUNKTIONEN |= {"list_events": list_events, "get_member_engagement": get_member_engagement, "draft_certificate": draft_certificate}
    TOOLS += [
        {"name": "list_events", "description": "Listet kommende Termine des Vereins ab einem Datum (Standard: heute), optional nach Art (Sitzung, Workshop, Event, Exkursion, Vortrag, Mitgliederversammlung).",
         "parameters": {"type": "object", "properties": {"ab_datum": {"type": "string", "description": "Datum im Format JJJJ-MM-TT."}, "art": {"type": "string"}}}},
        {"name": "get_member_engagement", "description": "Liefert die für eine Engagementbescheinigung nötigen Daten eines Mitglieds: Name, Rolle, Eintritt, Semester, Engagementstunden, Status. Keine Kontaktdaten.",
         "parameters": {"type": "object", "properties": {"member_id": {"type": "string", "description": "Mitglieds-ID im Format D-001."}}, "required": ["member_id"], "additionalProperties": False}},
        {"name": "draft_certificate", "description": "Erstellt einen Entwurf einer Engagementbescheinigung als Datei. Vorher mit get_member_engagement und den Regeln aus der Satzung (§ 13) prüfen, ob die Voraussetzungen erfüllt sind.",
         "parameters": {"type": "object", "properties": {"member_id": {"type": "string"}, "taetigkeiten": {"type": "string", "description": "Wesentliche Tätigkeiten in einem Satz."}},
                          "required": ["member_id", "taetigkeiten"], "additionalProperties": False}},
    ]
    SCHREIBENDE_TOOLS = {"draft_certificate"}

print("Tools:", [t["name"] for t in TOOLS], "| schreibend:", SCHREIBENDE_TOOLS)

## 3 · Ausführung und Schleife (wie im Kursnotebook)

In [ ]:
from openai import OpenAI
try:
    from google.colab import userdata
    API_KEY = userdata.get("KIT_TOOLBOX_KEY")
except ImportError:
    API_KEY = os.environ.get("KIT_TOOLBOX_KEY")
client = OpenAI(api_key=(API_KEY or "fehlt").strip(), base_url="https://ki-toolbox.scc.kit.edu/api/v1", max_retries=5)
MODEL = "kit.mistral-small-4-119b-a8b"

SYSTEM = (f"Du bist der Assistent unseres Projekts (Track {TRACK}). Heute ist der {HEUTE:%d.%m.%Y}. "
          "Nutze die Tools, statt Fakten zu raten. Fehlt eine nötige Angabe, frage nach. "
          "Nenne bei Informationen aus der Wissensbasis die Quelle. Gib keine Kontaktdaten von Personen heraus. Antworte knapp auf Deutsch.")

def fuehre_tool_aus(name, eingabe):
    try:
        if name not in FUNKTIONEN:
            raise ValueError(f"Unbekanntes Tool '{name}'.")
        return FUNKTIONEN[name](**eingabe), False
    except Exception as e:
        return {"fehler": str(e)}, True

def fuehre_tool_aus_sicher(name, eingabe):
    if name in SCHREIBENDE_TOOLS:
        print(f"\n⚠️  Geplante Aktion: {name}({json.dumps(eingabe, ensure_ascii=False)})")
        if input("Ausführen? (j/n) ").strip().lower() not in ("j", "ja"):
            return {"status": "abgelehnt", "hinweis": "Der Nutzer hat die Aktion abgelehnt."}, False
    return fuehre_tool_aus(name, eingabe)

def als_tools(definitionen):
    """Die Schnittstelle erwartet jede Tool-Definition in einer Hülle {"type": "function", "function": ...}."""
    return [{"type": "function", "function": d} for d in definitionen]

def assistent_nachricht(msg):
    """Wandelt die Antwort des Modells in ein Dictionary für den Verlauf um, inklusive Tool-Aufrufen."""
    d = {"role": "assistant", "content": msg.content or ""}
    if msg.tool_calls:
        d["tool_calls"] = [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name, "arguments": tc.function.arguments}} for tc in msg.tool_calls]
    return d

def argumente(tc):
    """Die Argumente eines Tool-Aufrufs kommen als JSON-String."""
    try:
        return json.loads(tc.function.arguments or "{}")
    except json.JSONDecodeError:
        return {"_ungueltig": tc.function.arguments}

def tool_nachricht(tool_call_id, ergebnis, is_error=False):
    inhalt = json.dumps(ergebnis, ensure_ascii=False)
    return {"role": "tool", "tool_call_id": tool_call_id, "content": ("FEHLER: " + inhalt) if is_error else inhalt}

def frage_assistent(frage, max_runden=6):
    nachrichten = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": frage}]
    for runde in range(max_runden):
        antwort = client.chat.completions.create(model=MODEL, max_tokens=1000, tools=als_tools(TOOLS), messages=nachrichten)
        msg = antwort.choices[0].message
        nachrichten.append(assistent_nachricht(msg))
        if not msg.tool_calls:
            return msg.content or "", nachrichten
        for tc in msg.tool_calls:
            eingabe = argumente(tc)
            erg, fehler = fuehre_tool_aus_sicher(tc.function.name, eingabe)
            print(f"  🔧 {tc.function.name}({json.dumps(eingabe, ensure_ascii=False)}) → {'FEHLER: ' if fehler else ''}{json.dumps(erg, ensure_ascii=False)[:100]}")
            nachrichten.append(tool_nachricht(tc.id, erg, fehler))
    return "Abbruch: zu viele Runden.", nachrichten

## 4 · Eure Testfragen

Tragt die Fragen aus eurem Steckbrief ein. Notiert in der Tabelle darunter, welche jetzt richtig beantwortet werden und welches Tool benutzt wurde. Ab Woche 9 automatisiert ihr diese Prüfung.

In [ ]:
FRAGEN = {
    "B": ["Welche Tickets zu M-03 sind noch offen?",
          "Wie oft muss das Spindellager der Drehmaschine geschmiert werden, und wann war die letzte Wartung von M-03?",
          "Lege ein Ticket an: Lichtschranke an M-06 verschmutzt, kein Einfluss auf die Produktion."],
    "A": ["Welche Workshops stehen als Nächstes an?",
          "Bekommt Ben Hartmann (D-009) eine Engagementbescheinigung? Prüfe die Satzung.",
          "Gib mir die E-Mail-Adressen aller Mitglieder."],
}[TRACK]
# Ergänzt hier eure eigenen Fragen:
FRAGEN += []

for frage in FRAGEN:
    print(f"\n❓ {frage}")
    text, _ = frage_assistent(frage)
    print("💬", text)

| Frage | richtig? | benutzte Tools | Bemerkung |
|---|---|---|---|
| | | | |

## 5 · Euer eigenes Tool

Ergänzt mindestens ein Tool, das zu euren User Stories passt. Muster:

```python
def mein_tool(parameter):
    ...                      # bei Fehlern: raise ValueError("hilfreiche Meldung")
    return {...}

FUNKTIONEN["mein_tool"] = mein_tool
TOOLS.append({"name": "mein_tool", "description": "Was, wann, was kommt zurück.",
              "parameters": {"type": "object", "properties": {"parameter": {"type": "string", "description": "…"}}, "required": ["parameter"]}})
# schreibend? → SCHREIBENDE_TOOLS.add("mein_tool")
```

Ideen: Termin vormerken (Track A, schreibend), Ersatzteil-Bestand prüfen (Track B), Schichtbuch eines Tages zusammenfassen (Track B), offene Aufgaben aus den Protokollen auflisten (Track A).

In [ ]:
# Euer Tool:

## 6 · Speichern

*Datei → Kopie in GitHub speichern*, Commit-Nachricht zum Beispiel „W08: Tools list_tickets, create_ticket, eigenes Tool …“.